## Updated Code

In [ ]:
from __future__ import annotations
import logging
import os
import random
import time

from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from typing import Any, Dict, Generator, Optional, Tuple, Union
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse

import requests
from dotenv import load_dotenv

load_dotenv()

logger = logging.getLogger(__name__)


DEFAULT_API_VERSION = "v27.0"
DEFAULT_BASE_URL = "https://graph.facebook.com"

DEFAULT_TIMEOUT = (5, 30)  # Connection/read timeout

DEFAULT_MAX_RETRIES = 4  # Maximum retries per request
DEFAULT_BACKOFF_FACTOR = 2.0  # Exponential backoff base
DEFAULT_MAX_BACKOFF = 60.0  # Maximum retry delay

RETRYABLE_HTTP_STATUS_CODES = {
    408,
    429,
    500,
    502,
    503,
    504,
}

RETRYABLE_GRAPH_ERROR_CODES = {
    1,  # Unknown API error
    2,  # Temporary API service error
}

# Meta rate-limit errors.
# These should normally stop automatic requests rather than repeatedly retrying and increasing API usage.
RATE_LIMIT_ERROR_CODES = {
    4,
    17,
    32,
    341,
    613,
    80001,
}

# used to track Meta Graph API usage
USAGE_HEADERS = (
    "X-App-Usage",  # Shows API usage against app's limits.
    "X-Page-Usage",  # Shows API usage associated with a Facebook Page.
    "X-Business-Use-Case-Usage",  # Shows usage for business-related API use cases.
)


class MetaGraphAPIError(Exception):
    """Structured exception for Meta Graph API failures."""

    def __init__(
        self,
        message: str,
        *,
        error_code: Optional[int] = None,
        error_subcode: Optional[int] = None,
        http_status: Optional[int] = None,
        error_type: Optional[str] = None,
        fbtrace_id: Optional[str] = None,
        is_transient: bool = False,
    ):
        super().__init__(message)

        self.message = message
        self.error_code = error_code
        self.error_subcode = error_subcode
        self.http_status = http_status
        self.error_type = error_type
        self.fbtrace_id = fbtrace_id
        self.is_transient = is_transient

    def __str__(self) -> str:
        details = []

        if self.error_code is not None:
            details.append(f"code={self.error_code}")

        if self.error_subcode is not None:
            details.append(f"subcode={self.error_subcode}")

        if self.http_status is not None:
            details.append(f"http_status={self.http_status}")

        suffix = f" ({', '.join(details)})" if details else ""

        return f"{self.message}{suffix}"


# Main Client for Meta Graph API
class FacebookCommentAnalyzerClient:
    """
    Client for interacting with the Meta Graph API.

    Features:
        - Access token authentication
        - Persistent HTTP connection pooling
        - Configurable connection/read timeouts
        - Bounded retries with exponential backoff and jitter
        - Retry-After support
        - Structured Graph API errors
        - Cursor-based pagination
        - Configurable page size and total result limits
        - Pagination URL validation
        - Usage header monitoring
        - Context manager support

    Environment variables:
        FACEBOOK_ACCESS_TOKEN
        PAGE_ID
    """

    BASE_URL = DEFAULT_BASE_URL

    def __init__(
        self,
        access_token: Optional[str] = None,
        api_version: str = DEFAULT_API_VERSION,
        timeout: Union[int, Tuple[float, float]] = DEFAULT_TIMEOUT,
        max_retries: int = DEFAULT_MAX_RETRIES,
        backoff_factor: float = DEFAULT_BACKOFF_FACTOR,
        max_backoff: float = DEFAULT_MAX_BACKOFF,
        pool_connections: int = 10,
        pool_maxsize: int = 10,
        user_agent: str = "FacebookCommentAnalyzer/1.0",
        retry_rate_limit_errors: bool = False,
    ):
        # Authentication
        self.access_token = access_token or os.getenv("FACEBOOK_ACCESS_TOKEN")

        if not self.access_token or not self.access_token.strip():
            raise ValueError(
                "No Facebook access token provided."
                "Set FACEBOOK_ACCESS_TOKEN in your environment."
            )

        # configure validation
        if not isinstance(api_version, str) or not api_version.startswith("v"):
            raise ValueError("Invalid Graph API version.")

        version_number = api_version[1:]
        if not version_number.replace(".", "").isdigit() or not version_number:
            raise ValueError("Invalid Graph API version.")

        if isinstance(timeout, (int, float)):
            timeout = (timeout, timeout)

        if (
            not isinstance(timeout, tuple)
            or len(timeout) != 2
            or any(
                not isinstance(value, (int, float)) or value <= 0 for value in timeout
            )
        ):
            raise ValueError(
                "timeout must be a positive number or a (connect_timeout, read_timeout) tuple"
            )

        if max_retries < 0:
            raise ValueError("max_retries cannot be negative.")

        if backoff_factor < 0:
            raise ValueError("backoff_factor cannot be negative.")

        if max_backoff <= 0:
            raise ValueError("max_backoff must be positive.")

        if pool_connections < 1 or pool_maxsize < 1:
            raise ValueError("Connection pool sizes must be positive.")

        self.api_version = api_version
        self.timeout = timeout
        self.max_retries = max_retries
        self.backoff_factor = backoff_factor
        self.max_backoff = max_backoff
        self.retry_rate_limit_errors = retry_rate_limit_errors

        # HTTP Session

        self.session = requests.sessions()

        adapter = requests.adapters.HTTPAdapter(
            pool_connections=pool_connections, pool_maxsize=pool_maxsize, max_retries=0
        )

        self.session.mount("httpsL//", adapter)

        self.session.headers.update(
            {
                "Accept": "Application/json",
                "User-Agent": user_agent,
            }
        )

        self._closed = False

    # Resource Management
    def close(self) -> None:
        """Close the HTTP session."""
        if not self._closed:
            self.session.close()
            self._closed = True

    def __enter__(self):
        if self._closed:
            raise RuntimeError("The client has already been closed.")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()

    # Validation
    @staticmethod
    def _validate_object_id(object_id: str, name: str) -> str:
        """Validate a Graph API object ID or endpoint segment."""

        if not isinstance(object_id, str):
            raise ValueError(f"{name} must be a string.")

        object_id = object_id.strip()

        if (
            not object_id
            or "/" in object_id
            or "?" in object_id
            or "#" in object_id
            or any(char.isspace() for char in object_id)
        ):
            raise ValueError(f"Invalid {name}.")
        return object_id

    @staticmethod
    def _validate_fields(fields: str) -> str:
        """Validate a Graph API fields parameter."""

        if not isinstance(fields, str) or not fields.strip():
            raise ValueError("fields must be a non-empty string.")

        return fields.strip()

    def _build_url(self, endpoint: str) -> str:
        """Build a versioned Graph API URL."""

        endpoint = self._validate_object_id(endpoint, "endpoint")

        return f"{self.BASE_URL}/{self.api_version}/{endpoint}"

    def _validate_pagination_url(self, url: str) -> None:
        """
        Validate URLs returned by Meta pagination.
        only HTTPS request to graph.facebook.com are allowed
        """

        try:
            parsed = urlparse(url)

            valid = (
                parsed.scheme == "https"
                and parsed.hostname == "graph.facebook.com"
                and parsed.post in (None, 443)
                and parsed.username is None
                and parsed.password is None
                and not parsed.fragment
                and parsed.path.startswith(f"/{self.api_version}")
            )
        except (TypeError, ValueError):
            valid = False

        if not valid:
            raise MetaGraphAPIError("Rejected an invalid or untrusted pagination URL.")

    # Response parsing
    def _parse_response(
        self,
        response: requests.Response,
    ) -> Dict[str, Any]:
        """Parse Graph API Json and HTTP errors."""
        try:
            data = response.json()

        except ValueError as exc:
            status = response.status_code

            raise MetaGraphAPIError(
                "Meta returned an invalid JSON response.",
                http_status=status,
                is_transient=(status in RETRYABLE_HTTP_STATUS_CODES or status >= 500),
            ) from exc

        if not isinstance(data, dict):
            raise MetaGraphAPIError(
                "Unexpected Graph API response format.",
                http_status=response.status_code,
            )

        # Graph API errors
        if "error" in data:
            error = data["error"]

            if not isinstance(error, dict):
                raise MetaGraphAPIError(
                    "Meta returned an invalid error payload.",
                    http_status=response.status_code,
                )

            code = error.get("code")
            subcode = error.get("error_subcode")

            is_rate_limit = code in RATE_LIMIT_ERROR_CODES

            is_transient = (
                code in RETRYABLE_GRAPH_ERROR_CODES
                or response.status_code in RETRYABLE_HTTP_STATUS_CODES
                or response.status_code >= 500
                or (bool(error.get("is_transient")) and not is_rate_limit)
            )

            raise MetaGraphAPIError(
                error.get("message", "Unknown Graph API error"),
                error_code=code,
                error_subcode=subcode,
                http_status=response.status_code,
                error_type=error.get("type"),
                fbtrace_id=error.get("fbtrace_id"),
                is_transient=is_transient,
            )

        # HTTP errors
        if not 200 <= response.status_code < 300:
            raise MetaGraphAPIError(
                f"HTTP request failed with status {response.status_code}.",
                http_status=response.status_code,
                is_transient=(
                    response.status_code in RETRYABLE_HTTP_STATUS_CODES
                    or response.status_code >= 500
                ),
            )
            return data

    # Usage Monitoring
    @staticmethod
    def _log_usage_headers(
        response: requests.Response,
    ) -> None:
        """Log Meta usage headers without exposing access tokens."""
        for header in USAGE_HEADERS:
            usage = response.headers.get(header)

            if usage:
                logger.info(
                    "Meta API usage [%s]: %s",
                    header,
                    usage,
                )

    # Retry handling
    def _get_retry_delay(
        self,
        attempt: int,
        response: Optional[requests.Response] = None,
    ) -> float:
        """Calculate retry delay with Retry-After and jitter."""

        if response is not None:
            retry_after = response.headers.get("Retry-After")

            if retry_after:
                try:
                    delay = float(retry_after)
                except ValueError:
                    try:
                        retry_data = parsedate_to_datetime(retry_after)

                        if retry_data.tzinfo is None:
                            retry_data = retry_data.replace(tzinfo=timezone.utc)
                        delay = (
                            retry_data - datetime.now(timezone.utc)
                        ).total_seconds()
                    except (TypeError, ValueError, OverflowError):
                        delay = -1

                if delay >= 0:
                    return min(delay, self.max_backoff)

        exponential_delay = self.backoff_factor * (2**attempt)

        exponential_delay = min(exponential_delay, self.max_backoff)
        return random.uniform(0, exponential_delay)

    def _should_retry(
        self,
        error: MetaGraphAPIError,
        attempt: int,
    ) -> bool:
        """Determine whether another request should be attempted."""

        if attempt >=self.max_retries:
            return False
        
        if error.error_code in RATE_LIMIT_ERROR_CODES:
            return self.retry_rate_limit_errors
        
        return error. is_transient


    # Core Request handler
    def _request_url(
        self,
        url: str,
        params: Optional[Dict[str,Any]] = None,
    ) -> Dict[str, Any]:
        """
        Execute a GET request with bounded retries.

        The access token is supplied as a query parameter, as supported by the Graph API.
        """

        if self._closed:
            raise RuntimeError("The Client is closed.")

        requests_params = dict(params or {})
        

_IncompleteInputError: incomplete input (1820073817.py, line 358)

```python
# Configure logging to show INFO and above in the terminal
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
```


```python
# in a file

import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("app.log"),      # Writes logs to app.log
        logging.StreamHandler(sys.stdout),   # Also prints logs to the terminal
    ],
)

```

In [ ]:
# import os
# import time
# import logging
# from typing import Dict, Any, Generator, Optional
# import requests
# from dotenv import load_dotenv

# load_dotenv()

# logger = logging.getLogger(__name__)


# class MetaGraphAPIError(Exception):
#     """Custom exception for Meta Graph API errors."""

#     def __init__(
#         self,
#         message: str,
#         error_code: Optional[int] = None,
#         error_subcode: Optional[int] = None,
#     ):
#         super().__init__(message)
#         self.error_code = error_code
#         self.error_subcode = error_subcode


# PAGE_ID = os.getenv("PAGE_ID")


# class FacebookCommentAnalyzerClient:
#     """Client for interacting with the Meta Graph API to fetch posts, comments, and replies."""

#     BASE_URL = "https://graph.facebook.com"

#     def __init__(
#         self,
#         access_token: Optional[str] = None,
#         api_version: str = "v27.0",
#         timeout: int = 30,
#     ):
#         """Initialize the client with an access token, API version, request timeout, and HTTP session."""
#         self.access_token = access_token or os.getenv("FACEBOOK_ACCESS_TOKEN")
#         if not self.access_token:
#             raise ValueError(
#                 "No Facebook Access Token provided or found in environment variables."
#             )
#         self.api_version = api_version
#         self.timeout = timeout
#         self.session = requests.Session()

#     def _parse_response(self, response: requests.Response) -> Dict[str, Any]:
#         """Parse a Graph API JSON response and raise MetaGraphAPIError if an error payload is present."""
#         data = response.json()

#         if "error" in data:
#             err = data["error"]
#             error_message = err.get("message", "Unknown Graph API error")
#             error_code = err.get("code")
#             error_subcode = err.get("error_subcode")

#             logger.error(
#                 f"Meta API Error [{error_code}] (Subcode: {error_subcode}): {error_message}"
#             )
#             raise MetaGraphAPIError(error_message, error_code, error_subcode)

#         return data

#     def _make_request(
#         self,
#         endpoint: str,
#         method: str = "GET",
#         params: Optional[Dict[str, Any]] = None,
#         json_data: Optional[Dict[str, Any]] = None,
#     ) -> Dict[str, Any]:
#         """Core request handler supporting GET, POST, DELETE, etc."""
#         url = f"{self.BASE_URL}/{self.api_version}/{endpoint}"

#         if params is None:
#             params = {}

#         # Always append access token
#         params["access_token"] = self.access_token
#         try:
#             # Dynamically call request.get, request.post, etc.
#             response = self.session.request(
#                 method=method.upper(),
#                 url=url,
#                 params=params if method.upper() == "GET" else None,
#                 json=json_data if method.upper() in ["POST", "PUT"] else None,
#                 timeout=self.timeout,
#             )
#             return self._parse_response(response)
#         except MetaGraphAPIError as e:
#             if e.error_code in [4, 17, 32]:
#                 logger.warning("Rate limit hit. Sleeping before retry...")
#                 time.sleep(10)
#                 return self._make_request(endpoint, method, params, json_data)
#             raise
#         except requests.exceptions.RequestException as e:
#             logger.exception("HTTP Request failed")
#             raise MetaGraphAPIError(
#                 f"Network error communicating with Meta API: {str(e)}"
#             )

#     def _fetch_raw_url(self, url: str) -> Dict[str, Any]:
#         """Fetch a full Graph API pagination URL directory with rate-limit retry handling."""
#         try:
#             response = self.session.get(url, timeout=self.timeout)
#             return self._parse_response(response)
#         except MetaGraphAPIError as e:
#             if e.error_code in [4, 7, 32]:
#                 logger.warning("Rate limit hit. Sleeping before retry...")
#                 time.sleep(10)
#                 return self._fetch_raw_url(url)
#             raise
#         except requests.exceptions.RequestException as e:
#             logger.exception("HTTP Request failed")
#             raise MetaGraphAPIError(
#                 f"Network error communicating with Meta API: {str(e)}"
#             )

#     def _paginate(
#         self, endpoint: str, params: Optional[Dict[str, Any]] = None
#     ) -> Generator[Dict[str, Any], None, None]:
#         """Yield items across pages by following Graph API paging.next URLs."""

#         next_url: Optional[str] = None
#         current_params: Optional[Dict[str, Any]] = params

#         while True:
#             if next_url:
#                 # Graph API 'next' links already include query params and access_token.
#                 response = self._fetch_raw_url(next_url)
#             else:
#                 response = self._make_request(
#                     endpoint, method="GET", params=current_params
#                 )
#                 current_params = None

#             for item in response.get("data", []):
#                 yield item

#             next_url = response.get("paging", {}).get("next")
#             if not next_url:
#                 break

#     def get_post_details(self, post_id: str) -> Dict[str, Any]:
#         """
#         Fetch metadata for a specific Facebook post or media object.
#         """
#         endpoint = f"{post_id}"
#         params = {
#             # "fields": "id,message,created_time,shares,comments.summary(true),likes.summary(true)"
#             "fields": "id,message,created_time,shares"
#         }
#         return self._make_request(endpoint, method="GET", params=params)

#     def get_all_posts(
#         self, page_id: Optional[str] = None, limit: int = 25
#     ) -> Generator[Dict[str, Any], None, None]:
#         """Yield posts from a Facebook Page using cursor-based pagination."""
#         target_page_id = page_id or os.getenv("PAGE_ID")
#         endpoint = f"{target_page_id}/posts"
#         if not target_page_id:
#             raise ValueError("No Page ID provided or found in environment variables.")

#         params = {
#             "fields": "id,message,created_time,shares",
#             "limit": limit,
#         }
#         return self._paginate(endpoint, params=params)

In [5]:
client = FacebookCommentAnalyzerClient()

In [6]:
for post in client.get_all_posts(limit=10):
    print(post)

{'id': '1403723142829428_122111281107533765', 'message': '27  post', 'created_time': '2026-11-19T18:15:19+0000'}
{'id': '1403723142829428_122111281059533765', 'message': '26  post', 'created_time': '2026-11-19T18:15:07+0000'}
{'id': '1403723142829428_122111281035533765', 'message': '25  post', 'created_time': '2026-11-19T18:14:57+0000'}
{'id': '1403723142829428_122111280987533765', 'message': '24 post', 'created_time': '2026-11-19T18:14:46+0000'}
{'id': '1403723142829428_122111280963533765', 'message': '23  post', 'created_time': '2026-11-19T18:14:34+0000'}
{'id': '1403723142829428_122111280897533765', 'message': '22 post', 'created_time': '2026-11-19T18:14:23+0000'}
{'id': '1403723142829428_122111280837533765', 'message': '21  post', 'created_time': '2026-11-19T18:14:10+0000'}
{'id': '1403723142829428_122111280789533765', 'message': '20 post', 'created_time': '2026-11-19T18:14:00+0000'}
{'id': '1403723142829428_122111280531533765', 'message': '19  post', 'created_time': '2026-11-19T18

In [7]:
client.get_post_details(post_id="1403723142829428_122106446457533765")


{'id': '1403723142829428_122106446457533765',
 'message': 'API TEST: Sample customer feedback thread. Please use this post only for testing the official Meta Pages API.',
 'created_time': '2026-11-09T05:48:18+0000',
 'comments': {'data': [{'created_time': '2026-11-09T05:48:37+0000',
    'from': {'name': 'NovaByte', 'id': '1403723142829428'},
    'message': 'TEST FEEDBACK: Delivery was late, but customer support was helpful. Please improve delivery tracking.',
    'id': '122106446457533765_2432515064184036'},
   {'created_time': '2026-11-19T17:58:55+0000',
    'from': {'name': 'NovaByte', 'id': '1403723142829428'},
    'message': '30 comment',
    'id': '122106446457533765_2961350974236005'},
   {'created_time': '2026-11-19T17:58:52+0000',
    'from': {'name': 'NovaByte', 'id': '1403723142829428'},
    'message': '29 comment',
    'id': '122106446457533765_1742389564169576'},
   {'created_time': '2026-11-19T17:58:48+0000',
    'from': {'name': 'NovaByte', 'id': '1403723142829428'},
    

<bound method Logger.debug of <Logger __main__ (WARNING)>>